# Music Sentiment as an Economic Signal

**Working research question:**  
Can shifts in popular music — especially sonic sentiment and genre composition — provide information that improves forecasts of future consumer sentiment?

### Current notebook status
This notebook is currently focused on:

1. Loading and auditing the Billboard Hot 100 data.
2. Loading the 550K Spotify-derived song metadata without the very large lyrics column.
3. Documenting known data-quality problems in the 550K dataset.
4. Engineering a more defensible broad genre taxonomy.
5. Building a canonical Spotify lookup when the same title + artist appears many times.
6. Running deterministic Billboard-to-song matching before any fuzzy matching.

### Important data decision
The 550K dataset's `year` field is **not treated as the authoritative timeline** because obvious date errors were found (for example, modern songs coded as 1900). The Billboard `Date` column will be the time/popularity backbone for the forecasting analysis.

Exploratory code that helped reach the current decisions is preserved near the bottom as commented-out "thought process" cells.


## 1. Imports and project paths

Only packages needed for the current data-preparation stage are imported below.

Packages for fuzzy matching, forecasting, diagnostics, and plotting are preserved as comments for later so the notebook does not load unnecessary dependencies yet.

In [1]:
import ast
import re
import unicodedata
from pathlib import Path

import pandas as pd

# --- Later stages: keep for reference, but not needed yet ---
# import numpy as np
# import matplotlib.pyplot as plt
#
# from rapidfuzz import fuzz, process
# from sklearn.metrics import mean_squared_error, mean_absolute_error
# from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
# from statsmodels.tsa.ar_model import AutoReg
# from statsmodels.tsa.statespace.sarimax import SARIMAX
# from statsmodels.stats.diagnostic import acorr_ljungbox

PROJECT_ROOT = Path.cwd()

# If the notebook is opened from a notebooks/ subfolder, move one level up.
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

BILLBOARD_PATH = DATA_RAW / "hot100.csv"
SONGS_PATH = DATA_RAW / "songs.csv"
SONG_METADATA_PATH = DATA_PROCESSED / "song_metadata.parquet"

print("Project root:", PROJECT_ROOT)

Project root: /Users/mercyeme/Documents/Data_Projects_2026/RecessionPop


## 2. Billboard Hot 100

The full Billboard table remains untouched as the weekly popularity/time table.

A separate `billboard_songs` table contains only unique `Song` + `Artist` pairs.  
This prevents repeatedly matching the same recording for every week it appears on the chart.

In [2]:
billboard = pd.read_csv(BILLBOARD_PATH)

print("Shape:", billboard.shape)
print("Columns:", billboard.columns.tolist())

billboard["Date"] = pd.to_datetime(billboard["Date"])

display(billboard.head())

print("First chart date:", billboard["Date"].min())
print("Last chart date:", billboard["Date"].max())

Shape: (354858, 8)
Columns: ['Date', 'Song', 'Artist', 'Rank', 'Last Week', 'Peak Position', 'Weeks in Charts', 'Image URL']


,Date,Song,Artist,Rank,Last Week,Peak Position,Weeks in Charts,Image URL
0,1958-07-30,Poor Little Fool,Ricky Nelson,1,-,1,1,NaN
1,1958-07-30,Patricia,Perez Prado And His Orchestra,2,-,2,1,NaN
2,1958-07-30,Splish Splash,Bobby Darin,3,-,3,1,NaN
3,1958-07-30,Hard Headed Woman,Elvis Presley With The Jordanaires,4,-,4,1,https://charts-static.billboard.com/img/1958/0...
4,1958-07-30,When,Kalin Twins,5,-,5,1,NaN


First chart date: 1958-07-30 00:00:00
Last chart date: 2026-07-29 00:00:00


In [3]:
billboard_songs = (
    billboard[["Song", "Artist"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"Weekly Billboard rows: {len(billboard):,}")
print(f"Unique song-artist pairs: {len(billboard_songs):,}")

display(billboard_songs.head())

Weekly Billboard rows: 354,858
Unique song-artist pairs: 32,592


,Song,Artist
0,Poor Little Fool,Ricky Nelson
1,Patricia,Perez Prado And His Orchestra
2,Splish Splash,Bobby Darin
3,Hard Headed Woman,Elvis Presley With The Jordanaires
4,When,Kalin Twins


## 3. Load lightweight 550K song metadata

The original `songs.csv` is very large because it contains full lyrics.

For the first pass, only metadata and audio-feature columns are loaded. If the processed Parquet file already exists, the notebook loads that instead of scanning the full CSV again.

**Lyrics stay out of memory until after Billboard matching.**

In [4]:
song_cols = [
    "id",
    "name",
    "album_name",
    "artists",
    "danceability",
    "energy",
    "key",
    "loudness",
    "mode",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "duration_ms",
    "year",
    "genre",
    "popularity",
    "total_artist_followers",
    "avg_artist_popularity",
    "artist_ids",
    "niche_genres",
]

In [5]:
if SONG_METADATA_PATH.exists():
    print("Loading processed metadata:", SONG_METADATA_PATH)
    songs = pd.read_parquet(SONG_METADATA_PATH)

else:
    print("Processed metadata not found; reading songs.csv in chunks.")

    chunks = []

    for chunk in pd.read_csv(
        SONGS_PATH,
        usecols=song_cols,
        chunksize=50_000,
    ):
        chunks.append(chunk)

    songs = pd.concat(chunks, ignore_index=True)

    songs.to_parquet(
        SONG_METADATA_PATH,
        index=False,
    )

print("Songs shape:", songs.shape)
display(songs.head(30))

Loading processed metadata: /Users/mercyeme/Documents/Data_Projects_2026/RecessionPop/data/processed/song_metadata.parquet
Songs shape: (550622, 23)


,id,name,album_name,artists,danceability,energy,key,loudness,mode,speechiness,...,valence,tempo,duration_ms,year,genre,popularity,total_artist_followers,avg_artist_popularity,artist_ids,niche_genres
0,0Prct5TDjAnEgIqbxcldY9,!,UNDEN!ABLE,"[""HELLYEAH""]",0.415,0.6050,7,-11.157,1,0.0575,...,0.193,100.059,79500,2016,Rock,0,769490,52.000000,"[""4hxDvVq5t8ebPYPdBl1F9f""]","[""groove metal"", ""metal""]"
1,2ASl4wirkeYm3OWZxXKYuq,!!,Childhood Dreams,"[""Yxngxr1""]",0.788,0.6480,7,-9.135,0,0.3150,...,0.287,79.998,114000,2019,Hip-Hop,29,143628,45.000000,"[""2jwRHcdgkRhelYEMqndDKe""]",[]
2,5tA3ImW310llKo8EMBj2Ga,!!Noble Stabbings!!,Situationist Comedy,"[""Dillinger Four""]",0.171,0.9570,2,-5.749,1,0.1490,...,0.349,175.317,197400,2002,Rock,0,36619,35.000000,"[""4YAN46l70QV0PGXlMg0iHi""]","[""melodic hardcore"", ""pop punk"", ""punk"", ""skat..."
3,0fROT4kK5oTm8xO8PX6EJF,!I'll Be Back!,!I'll Be Back!,"[""Ril\u00e8s""]",0.823,0.6120,1,-7.767,1,0.2480,...,0.688,142.959,178533,2018,Hip-Hop,43,929303,63.000000,"[""6pdcQa7by8IKuoVXvgknlI""]","[""french rap""]"
4,1xBFhv5faebv3mmwxx7DnS,!Lost!,!Lost!,"[""Ril\u00e8s""]",0.729,0.5520,7,-8.562,0,0.0650,...,0.380,86.103,186197,2018,Hip-Hop,0,929303,63.000000,"[""6pdcQa7by8IKuoVXvgknlI""]","[""french rap""]"
5,0gNNToCW3qjabgTyBSjt3H,!Que Vida! - Mono Version,Da Capo (Deluxe Edition),"[""Love""]",0.600,0.5400,9,-11.803,1,0.0328,...,0.547,125.898,220560,1966,Rock,20,273598,46.000000,"[""3Q6OOkfssqoMSTtl11J5Uk""]","[""acid rock"", ""baroque pop"", ""proto-punk"", ""ps..."
6,1SCFa6gt3r6juLsIqpdwY3,!liaF cipE,The Bree-TeenZ,"[""One Morning Left""]",0.483,0.8940,9,-4.164,1,0.0546,...,0.284,140.058,200200,2011,Rock,0,38169,34.000000,"[""4QGKWF55nqfGYToZkoZPRF""]","[""metalcore"", ""post-hardcore""]"
7,1JrpMwlzPuWgfmGiIBi9QI,""" """,Burner,"[""Breadwinner""]",0.296,0.8310,4,-8.101,1,0.0415,...,0.217,124.921,201760,1994,Rock,0,5014,16.000000,"[""3UUl93ABT7zywawX0QA7MC""]","[""math rock"", ""mathcore"", ""noise rock"", ""post-..."
8,5PIf0fmKJzgK8jgyqXaHGZ,""" R.I.P """,Escape,"[""Ram-Zet""]",0.474,0.9230,6,-4.611,1,0.1480,...,0.288,159.974,533160,2002,Rock,0,4100,7.000000,"[""41x3tK4pmcp109A09rYtxj""]","[""avant-garde"", ""gothic metal"", ""industrial me..."
9,0sZb4GBh64F0FOZzQy6QqV,"""...Because All Suffering Is Sweet to Me...""",Lucy Gray,"[""Envy On The Coast""]",0.241,0.7340,4,-6.919,0,0.0671,...,0.040,157.355,237973,2007,Rock,23,37014,31.000000,"[""6RvruFAgvsAb4B82u8wsSH""]","[""post-hardcore""]"


## 4. Data-quality notes

Two findings from exploration matter for the rest of the project:

- `year` contains obvious errors, including modern recordings assigned to 1900.
- The broad `genre` field is useful as a baseline, but some categories — especially `Folk` — contain systematic misclassifications such as reggae/dancehall.

Therefore:

- **Billboard `Date` controls time.**
- **Official broad genre is the fallback classification, not unquestioned ground truth.**
- `niche_genres` supplies evidence for targeted overrides.

In [6]:
print("550K year range:", songs["year"].min(), "to", songs["year"].max())
print("\nOfficial broad genre counts:")
display(songs["genre"].value_counts(dropna=False))

550K year range: 1900 to 2025

Official broad genre counts:


genre
Rock          197168
Pop            72539
Electronic     69755
Folk           50843
Country        46076
Hip-Hop        41745
R&B            27337
Jazz           17010
Blues          15976
Classical      12173
Name: count, dtype: int64

## 5. Parse niche genres

`niche_genres` is stored as a string representation of a Python list.  
Convert it once into an actual list so each tag can be inspected and scored safely.

In [7]:
def parse_genre_list(value):
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, list) else []

    except (ValueError, SyntaxError, TypeError):
        return []


songs["niche_genre_list"] = (
    songs["niche_genres"]
    .apply(parse_genre_list)
)

## 6. Final broad genre taxonomy

The current working taxonomy is intentionally broad enough for monthly genre-share features:

- Pop
- Indie
- Rock
- Hip-Hop
- R&B/Soul
- Country
- Electronic/Dance
- Folk
- Reggae/Dancehall
- Latin/Tropical
- Afrobeats/African Pop
- Jazz
- Classical/Musical
- Blues
- Other

### Classification philosophy

This is **not** a manual remapping of all 754 niche tags.

Instead:

1. Start with the dataset's official broad genre.
2. Rename broad labels where needed (`R&B` → `R&B/Soul`, etc.).
3. Let high-confidence niche tags vote for a corrected category.
4. If there is one clear winning category, use it as an override.
5. If niche evidence is absent or ambiguous, keep the mapped official genre.
6. `Other` is a fallback for official labels outside the defined taxonomy — not the automatic destination for an unmapped niche tag.

This preserves the original dataset where it behaves sensibly while correcting systematic problems.

In [8]:
FINAL_GENRES = [
    "Pop",
    "Indie",
    "Rock",
    "Hip-Hop",
    "R&B/Soul",
    "Country",
    "Electronic/Dance",
    "Folk",
    "Reggae/Dancehall",
    "Latin/Tropical",
    "Afrobeats/African Pop",
    "Jazz",
    "Classical/Musical",
    "Blues",
    "Other",
]

BASE_GENRE_MAP = {
    "Pop": "Pop",
    "Rock": "Rock",
    "Hip-Hop": "Hip-Hop",
    "R&B": "R&B/Soul",
    "Country": "Country",
    "Electronic": "Electronic/Dance",
    "Folk": "Folk",
    "Jazz": "Jazz",
    "Classical": "Classical/Musical",
    "Blues": "Blues",
}

songs["genre_original"] = songs["genre"]

songs["genre_base"] = (
    songs["genre_original"]
    .map(BASE_GENRE_MAP)
    .fillna("Other")
)

## 7. High-confidence niche-tag votes

These tags provide enough musical information to vote for one of the final broad categories.

A few deliberate choices:

- generic/geographic `indie` → **Indie**
- `indie rock` → **Rock**
- `indie r&b` → **R&B/Soul**
- `indie pop` → **Pop**
- `ska` → **Reggae/Dancehall**
- `ska punk` → **Rock**
- `pop punk` → **Rock**
- `synthpop` → **Electronic/Dance**
- generic `industrial`, `darkwave`, `new wave`, and `riddim` do **not** hard-override because they are genuinely cross-genre in this dataset.
- thematic/context tags such as `christian`, `christmas`, and `worship` do not vote for a primary musical genre.

In [9]:
GENRE_VOTE_TAGS = {

    "Reggae/Dancehall": {
        "calypso",
        "christian reggae",
        "dancehall",
        "dub",
        "lovers rock",
        "nz reggae",
        "ragga",
        "reggae",
        "reggae en español",
        "reggae pop",
        "reggae rock",
        "rocksteady",
        "roots reggae",
        "ska",
        "soca",
    },

    "Afrobeats/African Pop": {
        "afro adura",
        "afro r&b",
        "afro soul",
        "afrobeat",
        "afrobeats",
        "afrogospel",
        "afropiano",
        "afropop",
        "afroswing",
        "alté",
        "amapiano",
        "asakaa",
        "azonto",
        "bongo flava",
        "ghanaian hip hop",
        "highlife",
        "hiplife",
        "nigerian drill",
        "rumba congolaise",
    },

    "Latin/Tropical": {
        "bachata",
        "bolero",
        "bossa nova",
        "brazilian funk",
        "chilean mambo",
        "christian reggaeton",
        "corridos bélicos",
        "corridos tumbados",
        "cumbia",
        "cumbia norteña",
        "cumbia sonidera",
        "electro corridos",
        "electrocumbia",
        "forró",
        "forró tradicional",
        "funk carioca",
        "latin",
        "latin afrobeats",
        "latin alternative",
        "latin country",
        "latin dance",
        "latin folk",
        "latin folklore",
        "latin hip hop",
        "latin house",
        "latin indie",
        "latin jazz",
        "latin pop",
        "latin r&b",
        "latin rock",
        "mambo",
        "mariachi",
        "merengue",
        "mpb",
        "música mexicana",
        "new mpb",
        "norteño",
        "norteño-sax",
        "pagode",
        "pagode baiano",
        "ranchera",
        "reggaeton",
        "reggaeton chileno",
        "reggaeton mexa",
        "salsa",
        "samba",
        "sertanejo",
        "sertanejo tradicional",
        "sertanejo universitário",
        "tango",
        "trap latino",
        "urbano latino",
    },

    "Indie": {
        "indie",
        "chinese indie",
        "german indie",
        "hindi indie",
        "indian indie",
        "indie québécois",
        "indonesian indie",
        "japanese indie",
        "lo-fi indie",
        "mexican indie",
        "pinoy indie",
        "taiwanese indie",
        "tamil indie",
        "vietnam indie",
    },

    "R&B/Soul": {
        "alternative r&b",
        "classic soul",
        "dark r&b",
        "funk",
        "indie r&b",
        "indie soul",
        "motown",
        "neo soul",
        "new jack swing",
        "northern soul",
        "philly soul",
        "quiet storm",
        "r&b",
        "retro soul",
        "soul",
        "trap soul",
    },

    "Hip-Hop": {
        "alternative hip hop",
        "boom bap",
        "cloud rap",
        "crunk",
        "drill",
        "east coast hip hop",
        "emo rap",
        "experimental hip hop",
        "g-funk",
        "gangster rap",
        "hardcore hip hop",
        "hip hop",
        "melodic rap",
        "old school hip hop",
        "rap",
        "southern hip hop",
        "trap",
        "underground hip hop",
        "west coast hip hop",
    },

    "Electronic/Dance": {
        "deep house",
        "disco",
        "drum and bass",
        "dubstep",
        "ebm",
        "edm",
        "electro house",
        "electronic",
        "eurodance",
        "future bass",
        "hardstyle",
        "house",
        "melodic bass",
        "progressive house",
        "synthpop",
        "tech house",
        "techno",
        "trance",
        "tropical house",
    },

    "Country": {
        "alt country",
        "americana",
        "bluegrass",
        "classic country",
        "country",
        "honky tonk",
        "outlaw country",
        "red dirt",
        "texas country",
        "traditional country",
    },

    "Jazz": {
        "bebop",
        "big band",
        "cool jazz",
        "jazz",
        "jazz funk",
        "jazz fusion",
        "smooth jazz",
        "soul jazz",
        "swing music",
        "vocal jazz",
    },

    "Blues": {
        "blues",
        "classic blues",
        "country blues",
        "delta blues",
        "modern blues",
    },

    "Classical/Musical": {
        "baroque",
        "chamber music",
        "choral",
        "classical",
        "classical crossover",
        "contemporary classical",
        "musicals",
        "opera",
        "orchestral",
        "renaissance",
    },

    "Folk": {
        "anti-folk",
        "celtic folk",
        "folk",
        "folk pop",
        "indie folk",
        "singer-songwriter",
        "traditional folk",
    },

    "Rock": {
        "alternative rock",
        "art rock",
        "blues rock",
        "classic rock",
        "death metal",
        "funk rock",
        "garage rock",
        "glam rock",
        "grunge",
        "hard rock",
        "hardcore",
        "indie rock",
        "industrial metal",
        "industrial rock",
        "metal",
        "metalcore",
        "pop punk",
        "post-grunge",
        "post-hardcore",
        "power metal",
        "progressive rock",
        "psychedelic rock",
        "psychobilly",
        "punk",
        "rock",
        "rock and roll",
        "ska punk",
        "skate punk",
        "soft rock",
        "yacht rock",
    },

    "Pop": {
        "art pop",
        "baroque pop",
        "bedroom pop",
        "dance pop",
        "electropop",
        "funk pop",
        "indie pop",
        "j-pop",
        "k-pop",
        "pop",
        "power pop",
        "soft pop",
    },
}

# Additional explicit genre tags discovered during audit

GENRE_VOTE_TAGS["Rock"].update({
    "christian rock",
    "christian alternative rock",
    "southern rock",
    "celtic rock",
    "j-rock",
    "krautrock",
})

GENRE_VOTE_TAGS["Country"].update({
    "country christian",
    "christian bluegrass",
    "southern gospel",
})

GENRE_VOTE_TAGS["Folk"].update({
    "christian folk",
    "celtic",
})

GENRE_VOTE_TAGS["Pop"].update({
    "christian pop",
    "pop worship",
    "dream pop",
    "acoustic pop",
})

GENRE_VOTE_TAGS["R&B/Soul"].update({
    "christian r&b",
    "uk r&b",
})

GENRE_VOTE_TAGS["Hip-Hop"].update({
    "christian hip hop",
    "grime",
    "uk grime",
    "chinese hip hop",
    "k-rap",
    "miami bass",
})

GENRE_VOTE_TAGS["Electronic/Dance"].update({
    "alternative dance",
    "electro swing",
    "synthwave",
    "witch house",
})

GENRE_VOTE_TAGS["Jazz"].update({
    "acid jazz",
    "experimental jazz",
})

GENRE_VOTE_TAGS["Classical/Musical"].update({
    "gregorian chant",
})

# Additional religious subgenre votes that still preserve the separate
# religious_music flag.

GENRE_VOTE_TAGS["Jazz"].update({
    "christian jazz",
})

GENRE_VOTE_TAGS["R&B/Soul"].update({
    "funk gospel",
})

GENRE_VOTE_TAGS["Latin/Tropical"].update({
    "brazilian gospel",
    "pagode gospel",
    "sertanejo gospel",
})

# Tags that are meaningful but too cross-genre to force a primary classification.

INHERIT_TAGS = {
    "industrial",
    "darkwave",
    "new wave",
    "riddim",
    "folk rock",
    "country rock",
    "gospel",
    "traditional gospel",
}

# Descriptors/context tags that should not vote for a primary musical genre.

PRIMARY_GENRE_IGNORE_TAGS = {
    "christian",
    "christmas",
    "worship",
    "ccm",
    "hymns",
    "soundtrack",
    "children's music",
    "comedy",
}

### Sanity check: a niche tag should not vote for multiple categories

This catches accidental duplicates in the manual vote sets before classification runs.

In [10]:
tag_to_categories = {}

for category, tags in GENRE_VOTE_TAGS.items():
    for tag in tags:
        tag_to_categories.setdefault(tag, []).append(category)

duplicate_vote_tags = {
    tag: categories
    for tag, categories in tag_to_categories.items()
    if len(categories) > 1
}

assert not duplicate_vote_tags, (
    "Some niche tags vote for multiple categories: "
    f"{duplicate_vote_tags}"
)

print(
    "Unique high-confidence voting tags:",
    len(tag_to_categories),
)

Unique high-confidence voting tags: 273


In [11]:
vote_tags = set().union(*GENRE_VOTE_TAGS.values())

rule_conflicts = {
    "vote_and_inherit": vote_tags & INHERIT_TAGS,
    "vote_and_ignore": vote_tags & PRIMARY_GENRE_IGNORE_TAGS,
    "inherit_and_ignore": INHERIT_TAGS & PRIMARY_GENRE_IGNORE_TAGS,
}

rule_conflicts = {
    name: tags
    for name, tags in rule_conflicts.items()
    if tags
}

assert not rule_conflicts, (
    f"Conflicting genre rules found: {rule_conflicts}"
)

print("No conflicting genre rules.")

No conflicting genre rules.


## 8. Score niche tags and apply primary-genre overrides

Every exact tag match contributes one vote to its broad category.

Why exact matching? Substring matching causes false positives such as:

- `ska` matching `skate punk`
- `dub` matching `dubstep`
- `rap` matching `trap`

If multiple categories tie for the highest score, the notebook keeps the mapped official genre rather than forcing a questionable override.

In [12]:
def genre_scores(tags):
    tag_set = {
        str(tag).lower().strip()
        for tag in tags
        if pd.notna(tag)
    }

    return {
        category: len(tag_set & category_tags)
        for category, category_tags in GENRE_VOTE_TAGS.items()
    }


def choose_genre_override(tags):
    scores = genre_scores(tags)

    best_score = max(scores.values())

    if best_score == 0:
        return None

    winners = [
        category
        for category, score in scores.items()
        if score == best_score
    ]

    # Ambiguous niche evidence: keep official/base genre.
    if len(winners) != 1:
        return None

    return winners[0]

In [13]:
songs["genre_override"] = (
    songs["niche_genre_list"]
    .apply(choose_genre_override)
)

songs["genre_engineered"] = (
    songs["genre_override"]
    .fillna(songs["genre_base"])
)

unexpected_genres = (
    set(songs["genre_engineered"].dropna().unique())
    - set(FINAL_GENRES)
)

assert not unexpected_genres, (
    f"Unexpected engineered genres: {unexpected_genres}"
)

## 8A. Create secondary thematic features

Some niche tags describe themes, communities, or contexts rather than a
primary musical genre.

Religious tags therefore do not determine `genre_engineered`, but are
preserved as a separate binary feature.

This allows a song to be, for example:

- `genre_engineered = "Hip-Hop"`
- `religious_music = 1`

rather than forcing "Christian" to become a primary genre.

In [14]:
RELIGIOUS_TAGS = {
    "christian",
    "christian alternative rock",
    "christian bluegrass",
    "christian folk",
    "christian hip hop",
    "christian jazz",
    "christian lo-fi",
    "christian pop",
    "christian r&b",
    "christian reggae",
    "christian reggaeton",
    "christian rock",
    "country christian",
    "ccm",
    "gospel",
    "traditional gospel",
    "southern gospel",
    "afrogospel",
    "brazilian gospel",
    "funk gospel",
    "pagode gospel",
    "sertanejo gospel",
    "worship",
    "pop worship",
    "hymns",
    "catholic music"
}

In [15]:
def has_religious_tag(tags):
    tag_set = {
        str(tag).lower().strip()
        for tag in tags
        if pd.notna(tag)
    }

    return int(bool(tag_set & RELIGIOUS_TAGS))


songs["religious_music"] = (
    songs["niche_genre_list"]
    .apply(has_religious_tag)
)

In [16]:
print(
    songs["religious_music"]
    .value_counts(dropna=False)
)

print(
    f"Religiously tagged songs: "
    f"{songs['religious_music'].sum():,}"
)

religious_music
0    520298
1     30324
Name: count, dtype: int64
Religiously tagged songs: 30,324


In [17]:
display(
    songs.loc[
        songs["religious_music"] == 1,
        [
            "name",
            "artists",
            "genre_original",
            "genre_engineered",
            "niche_genres",
        ]
    ].sample(30, random_state=42)
)

,name,artists,genre_original,genre_engineered,niche_genres
241320,It Is Well (Live),"[""Bethel Music"", ""Kristene DiMarco""]",Pop,Pop,"[""ccm"", ""christian"", ""pop worship"", ""worship""]"
212337,I Fade Away,"[""Pillar""]",Rock,Rock,"[""chinese rock"", ""christian alternative rock"",..."
72204,Burning Bridges,"[""Seventh Day Slumber""]",Rock,Rock,"[""ccm"", ""chinese rock"", ""christian"", ""christia..."
147223,Farther Along (Live),"[""The Petersens""]",Country,Country,"[""bluegrass"", ""christian bluegrass"", ""christia..."
176039,Gone Country,"[""Alan Jackson""]",Country,Country,"[""classic country"", ""country"", ""country christ..."
180083,Gravity,"[""John Schlitt""]",Rock,Rock,"[""ccm"", ""christian alternative rock"", ""christi..."
199713,Holding Onto Hope,"[""I AM THEY""]",Folk,Folk,"[""ccm"", ""christian"", ""christian folk"", ""worship""]"
218664,I Then Shall Live - A Cappella,"[""Gaither Vocal Band""]",Country,Country,"[""christian"", ""christian bluegrass"", ""country ..."
389729,Searching - Stripped Version,"[""Jlyricz""]",Pop,Afrobeats/African Pop,"[""afrogospel"", ""afropop""]"
403627,Sinking Stone,"[""Alison Krauss & Union Station""]",Country,Country,"[""bluegrass"", ""christian bluegrass"", ""newgrass""]"


## 9. Audit the engineered result

Keep an audit trail rather than replacing the source columns.

Useful columns:

- `genre_original`: source dataset's broad genre
- `genre_base`: source genre renamed into the final taxonomy
- `niche_genres`: original niche-tag string
- `genre_override`: niche-tag winner, if there was one
- `genre_engineered`: final category used for analysis

In [18]:
genre_summary = (
    songs["genre_engineered"]
    .value_counts(dropna=False)
    .rename_axis("genre_engineered")
    .reset_index(name="song_count")
)

display(genre_summary)

,genre_engineered,song_count
0,Rock,207008
1,Electronic/Dance,67840
2,Pop,58224
3,Country,46178
4,Hip-Hop,40467
5,R&B/Soul,32787
6,Folk,31818
7,Jazz,17846
8,Reggae/Dancehall,15770
9,Blues,11923


In [19]:
changed_genres = songs.loc[
    songs["genre_engineered"] != songs["genre_base"],
    [
        "name",
        "artists",
        "genre_original",
        "genre_base",
        "niche_genres",
        "genre_override",
        "genre_engineered",
    ],
]

print(f"Rows changed by niche-tag overrides: {len(changed_genres):,}")

if len(changed_genres) > 0:
    display(
        changed_genres.sample(
            min(40, len(changed_genres)),
            random_state=42,
        )
    )

Rows changed by niche-tag overrides: 69,518


,name,artists,genre_original,genre_base,niche_genres,genre_override,genre_engineered
480102,Throne,"[""Original Koffee""]",Folk,Folk,"[""afrobeats"", ""dancehall"", ""reggae""]",Reggae/Dancehall,Reggae/Dancehall
49014,Bebe,"[""2Baba"", ""Larry Gaaga"", ""Mi Casa""]",R&B,R&B/Soul,"[""afro house"", ""afro r&b"", ""afrobeat"", ""afrobe...",Afrobeats/African Pop,Afrobeats/African Pop
175513,Gold to Me,"[""Dirty Heads""]",Folk,Folk,"[""reggae"", ""reggae rock""]",Reggae/Dancehall,Reggae/Dancehall
178798,Got Me,"[""Roy Woods""]",Hip-Hop,Hip-Hop,"[""trap soul""]",R&B/Soul,R&B/Soul
425169,Still Pt. 3,"[""A-One"", ""K Koke""]",Electronic,Electronic/Dance,"[""grime"", ""uk grime""]",Hip-Hop,Hip-Hop
298272,Miley Cyrus (Twerk),"[""Elephant Man""]",Folk,Folk,"[""dancehall"", ""ragga"", ""reggae"", ""riddim"", ""so...",Reggae/Dancehall,Reggae/Dancehall
318028,New Mess,"[""Helvetia""]",Rock,Rock,"[""lo-fi indie"", ""shoegaze"", ""slowcore""]",Indie,Indie
507145,Walking in the Air,"[""Howard Blake"", ""Laura Auer"", ""Broadway Studi...",Pop,Pop,"[""christmas"", ""classical crossover""]",Classical/Musical,Classical/Musical
249138,Jumbo,"[""Ice Prince""]",R&B,R&B/Soul,"[""afrobeat"", ""afrobeats"", ""afropop"", ""azonto"",...",Afrobeats/African Pop,Afrobeats/African Pop
508761,Wash Away,"[""Stone Senses""]",Folk,Folk,"[""reggae"", ""reggae rock""]",Reggae/Dancehall,Reggae/Dancehall


## 10. Tag-rule coverage audit

There are 754 unique niche tags in this dataset, but they do **not** all need a manual category.

This audit distinguishes:

- `vote` — explicitly supplies evidence for a category
- `inherit` — deliberately cross-genre; use the official/base genre unless other tags resolve it
- `ignore` — thematic/contextual; does not vote
- `fallback_to_official` — not manually mapped because the base broad genre remains the default

### `vote`

- This niche tag actively contributes a vote toward a primary genre.
- `manual_vote_category` is populated.

### `inherit`

- Deliberately cross-genre.
- This tag itself does not vote.
- Another tag on the same song can still cause an override.
- Otherwise, keep `genre_base`.

### `ignore`

- Deliberately not a primary-genre descriptor.
- Examples: `christian`, `christmas`, `worship`.
- Does not vote, but can still be used for a separate feature.

### `fallback_to_official`

- We have not assigned this niche tag a manual rule.
- It does not vote.
- If no other tag votes, keep `genre_base`.

This means every niche tag is covered by the **classification logic**, even though only a subset requires an explicit override.

In [20]:
all_niche_tags = set(
    songs["niche_genre_list"]
    .explode()
    .dropna()
    .astype(str)
    .str.lower()
    .str.strip()
)

vote_tags = set(tag_to_categories)

def get_rule_type(tag):
    if tag in vote_tags:
        return "vote"
    if tag in INHERIT_TAGS:
        return "inherit"
    if tag in PRIMARY_GENRE_IGNORE_TAGS:
        return "ignore"
    return "fallback_to_official"


tag_rule_summary = pd.Series(
    [get_rule_type(tag) for tag in all_niche_tags]
).value_counts()

print("Total niche genres:", len(all_niche_tags))
display(tag_rule_summary.rename("tag_count"))

Total niche genres: 754


fallback_to_official    467
vote                    271
inherit                   8
ignore                    8
Name: tag_count, dtype: int64

## 11. Optional: inspect the saved 754-tag audit

The previously generated `genre_tag_audit.csv` is useful for reviewing high-volume, low-agreement tags without rebuilding the file every time.

Low agreement is a **review flag**, not proof that a tag needs an override. Some tags genuinely span genres.

In [21]:
GENRE_AUDIT_PATH = DATA_PROCESSED / "genre_tag_audit.csv"

if GENRE_AUDIT_PATH.exists():
    genre_audit = pd.read_csv(GENRE_AUDIT_PATH)

    review_candidates = (
        genre_audit[
            (genre_audit["total_count"] >= 500)
            & (genre_audit["dominant_share"] < 0.80)
        ]
        .copy()
    )

    review_candidates["rule_type"] = (
        review_candidates["niche_genre"]
        .map(get_rule_type)
    )

    review_candidates["manual_vote_category"] = (
        review_candidates["niche_genre"]
        .map(
            {
                tag: category
                for category, tags in GENRE_VOTE_TAGS.items()
                for tag in tags
            }
        )
    )

    display(
        review_candidates[
            [
                "niche_genre",
                "total_count",
                "dominant_genre",
                "dominant_share",
                "second_genre",
                "second_share",
                "rule_type",
                "manual_vote_category",
                "examples",
            ]
        ].sort_values("total_count", ascending=False)
    )

else:
    print(
        "genre_tag_audit.csv not found. "
        "The rebuild code is preserved in the appendix."
    )

,niche_genre,total_count,dominant_genre,dominant_share,second_genre,second_share,rule_type,manual_vote_category,examples
8,pop punk,13001,Rock,0.598185,Pop,0.363280,vote,Rock,"!!Noble Stabbings!! — [""Dillinger Four""] | ""Co..."
14,blues rock,11499,Blues,0.788155,Rock,0.201670,vote,Rock,"""B"" Movie Box Car Blues - Live — [""The Blues B..."
16,industrial,10991,Rock,0.705395,Electronic,0.290601,inherit,NaN,"""President Dead"" — [""Marilyn Manson""] | $10 Bi..."
18,christian,10761,Pop,0.394666,Rock,0.300158,ignore,NaN,"#LETSGO - Live — [""Planetshakers""] | #LITO — [..."
20,ska punk,10346,Folk,0.520588,Rock,0.468200,vote,Rock,"#NotRealLife — [""H2O""] | $1000 Bender — [""Get ..."
...,...,...,...,...,...,...,...,...,...
362,miami bass,594,Electronic,0.629630,Hip-Hop,0.190236,vote,Hip-Hop,"2 Live Blues — [""2 LIVE CREW""] | 2 Live Is Her..."
363,soundtrack,590,Classical,0.666102,Pop,0.193220,ignore,NaN,"""DEVILS NEVER CRY""(スタッフロール) — [""Capcom Sound T..."
365,cajun,583,Blues,0.651801,Country,0.140652,fallback_to_official,NaN,"(Everybody Wanna Get Rich) Rite Away — [""Dr. J..."
369,dark ambient,556,Electronic,0.491007,Rock,0.483813,fallback_to_official,NaN,(Don’t Fear) the Reaper (Blue Oyster Cult) — [...


## 12. Save the engineered metadata

This saved file becomes the lightweight song-level lookup table for the matching stage.

The original `songs.csv` remains untouched.

In [22]:
ENGINEERED_METADATA_PATH = (
    DATA_PROCESSED / "song_metadata_engineered.parquet"
)

songs.to_parquet(
    ENGINEERED_METADATA_PATH,
    index=False,
)

print("Saved:", ENGINEERED_METADATA_PATH)

Saved: /Users/mercyeme/Documents/Data_Projects_2026/RecessionPop/data/processed/song_metadata_engineered.parquet


## 13. Billboard ↔ 550K song matching

The genre work is now treated as a **frozen first-pass taxonomy** unless a later audit reveals a serious systematic error.

The matching pipeline is deliberately staged:

1. normalize titles and artists,
2. collapse repeated Spotify copies into one canonical row per normalized title + primary artist,
3. run a strict exact-title + exact-artist pass,
4. run a second deterministic pass for Billboard credits written as `Lead Artist Featuring ...`,
5. inspect what remains before attempting aliases, `With`, or fuzzy matching.

The goal is to keep every accepted match **auditable** through a `match_method` column rather than silently relaxing the rules.


### 13.1 Normalize titles and artist names

Normalization is used only for matching keys. The original song and artist strings remain unchanged.

Important behavior:

- lowercase and trim whitespace,
- remove accents,
- convert `&` to `and`,
- remove punctuation,
- collapse repeated spaces,
- keep version words such as `live`, `remastered`, and `remix`.


In [23]:
def normalize_text(value):
    if pd.isna(value):
        return ""

    value = str(value).lower().strip()
    value = unicodedata.normalize("NFKD", value)

    value = "".join(
        char
        for char in value
        if not unicodedata.combining(char)
    )

    value = value.replace("&", "and")
    value = re.sub(r"[^\w\s]", " ", value)
    value = re.sub(r"\s+", " ", value)

    return value.strip()


def parse_artist_list(value):
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, list) else []

    except (ValueError, SyntaxError, TypeError):
        return []


In [24]:
billboard_songs["title_key"] = (
    billboard_songs["Song"]
    .map(normalize_text)
)

billboard_songs["artist_key"] = (
    billboard_songs["Artist"]
    .map(normalize_text)
)

songs["artist_list"] = (
    songs["artists"]
    .apply(parse_artist_list)
)

songs["primary_artist"] = (
    songs["artist_list"]
    .str[0]
)

songs["title_key"] = (
    songs["name"]
    .map(normalize_text)
)

songs["artist_key"] = (
    songs["primary_artist"]
    .map(normalize_text)
)


In [25]:
print(
    "Blank Spotify title keys:",
    songs["title_key"].eq("").sum()
)

print(
    "Blank Spotify artist keys:",
    songs["artist_key"].eq("").sum()
)

print(
    "Blank Billboard title keys:",
    billboard_songs["title_key"].eq("").sum()
)

print(
    "Blank Billboard artist keys:",
    billboard_songs["artist_key"].eq("").sum()
)

display(
    billboard_songs[
        [
            "Song",
            "Artist",
            "title_key",
            "artist_key",
        ]
    ].head(10)
)


Blank Spotify title keys: 38
Blank Spotify artist keys: 72
Blank Billboard title keys: 0
Blank Billboard artist keys: 0


,Song,Artist,title_key,artist_key
0,Poor Little Fool,Ricky Nelson,poor little fool,ricky nelson
1,Patricia,Perez Prado And His Orchestra,patricia,perez prado and his orchestra
2,Splish Splash,Bobby Darin,splish splash,bobby darin
3,Hard Headed Woman,Elvis Presley With The Jordanaires,hard headed woman,elvis presley with the jordanaires
4,When,Kalin Twins,when,kalin twins
5,Rebel-'rouser,Duane Eddy His Twangy Guitar And The Rebels,rebel rouser,duane eddy his twangy guitar and the rebels
6,Yakety Yak,The Coasters,yakety yak,the coasters
7,My True Love,Jack Scott,my true love,jack scott
8,Willie And The Hand Jive,The Johnny Otis Show,willie and the hand jive,the johnny otis show
9,Fever,Peggy Lee,fever,peggy lee


### 13.2 Diagnose repeated Spotify title + artist keys

The 550K source often repeats the same recording across original albums, compilations, and reissues.

A duplicate key is therefore **not automatically a conflict**. Before matching Billboard, repeated Spotify rows are collapsed into one canonical row using an audio-profile heuristic.


In [26]:
spotify_key_counts = (
    songs.loc[
        songs["title_key"].ne("")
        & songs["artist_key"].ne("")
    ]
    .groupby(
        [
            "title_key",
            "artist_key",
        ]
    )
    .size()
    .reset_index(name="candidate_count")
)

print(
    "Unique Spotify title-artist keys:",
    len(spotify_key_counts)
)

print(
    "Keys with multiple Spotify rows:",
    spotify_key_counts["candidate_count"].gt(1).sum()
)

display(
    spotify_key_counts.loc[
        spotify_key_counts["candidate_count"] > 1
    ]
    .sort_values(
        "candidate_count",
        ascending=False
    )
    .head(20)
)


Unique Spotify title-artist keys: 469176
Keys with multiple Spotify rows: 58770


,title_key,artist_key,candidate_count
108996,drive,the cars,70
323039,rockin around the christmas tree,brenda lee,45
232794,little saint nick 1991 remix,the beach boys,42
326033,run rudolph run,chuck berry,41
139229,frosty the snowman,ella fitzgerald,38
347827,sleigh ride,ella fitzgerald,33
190312,i saw mommy kissing santa claus,the jackson 5,29
433279,waiting for a girl like you,foreigner,28
421383,true,spandau ballet,28
225694,let it snow let it snow let it snow,dean martin,27


### 13.3 Build one canonical Spotify row per title + artist

Two manual checks motivated the rule:

- `Drive — The Cars` had 70 rows, with a dominant audio profile appearing 37 times; within that profile, the highest-popularity row was the `Heartbeat City` copy.
- `Last Last — Burna Boy` had 22 rows, with one profile appearing 21 times; within that profile, the highest-popularity row was the `Love, Damini` copy.

The production rule is therefore:

> Within each normalized title + artist group, prefer the most common rounded audio signature. Within that signature, prefer the highest-popularity row. Use Spotify ID only as a stable final tie-breaker.

The original audio-feature columns remain unchanged. Rounded `sig_*` columns exist only for duplicate resolution.


In [27]:
match_cols = [
    "id",
    "name",
    "album_name",
    "artists",
    "primary_artist",
    "title_key",
    "artist_key",
    "year",
    "popularity",
    "valence",
    "energy",
    "danceability",
    "tempo",
    "mode",
    "duration_ms",
    "genre_original",
    "genre_engineered",
    "religious_music",
]

spotify_lookup = (
    songs[match_cols]
    .copy()
)

spotify_lookup["sig_danceability"] = (
    spotify_lookup["danceability"]
    .round(3)
)

spotify_lookup["sig_energy"] = (
    spotify_lookup["energy"]
    .round(3)
)

spotify_lookup["sig_valence"] = (
    spotify_lookup["valence"]
    .round(3)
)

spotify_lookup["sig_tempo"] = (
    spotify_lookup["tempo"]
    .round(1)
)

spotify_lookup["sig_duration"] = (
    spotify_lookup["duration_ms"]
    .div(1000)
    .round()
)

signature_cols = [
    "sig_danceability",
    "sig_energy",
    "sig_valence",
    "sig_tempo",
    "sig_duration",
]


In [28]:
signature_counts = (
    spotify_lookup.loc[
        spotify_lookup["title_key"].ne("")
        & spotify_lookup["artist_key"].ne("")
    ]
    .groupby(
        [
            "title_key",
            "artist_key",
        ]
        + signature_cols,
        dropna=False,
    )
    .size()
    .reset_index(name="signature_count")
)

spotify_lookup = (
    spotify_lookup
    .merge(
        signature_counts,
        on=[
            "title_key",
            "artist_key",
        ]
        + signature_cols,
        how="left",
        validate="many_to_one",
    )
)

spotify_lookup = (
    spotify_lookup
    .merge(
        spotify_key_counts,
        on=[
            "title_key",
            "artist_key",
        ],
        how="left",
        validate="many_to_one",
    )
)

spotify_lookup["signature_share"] = (
    spotify_lookup["signature_count"]
    / spotify_lookup["candidate_count"]
)


In [29]:
spotify_exact_candidates = (
    spotify_lookup.loc[
        spotify_lookup["title_key"].ne("")
        & spotify_lookup["artist_key"].ne("")
    ]
    .copy()
)

spotify_exact_candidates["popularity_sort"] = (
    spotify_exact_candidates["popularity"]
    .fillna(-1)
)

spotify_exact_canonical = (
    spotify_exact_candidates
    .sort_values(
        [
            "title_key",
            "artist_key",
            "signature_count",
            "popularity_sort",
            "id",
        ],
        ascending=[
            True,
            True,
            False,
            False,
            True,
        ],
    )
    .drop_duplicates(
        subset=[
            "title_key",
            "artist_key",
        ],
        keep="first",
    )
    .drop(columns="popularity_sort")
    .reset_index(drop=True)
)

assert not spotify_exact_canonical.duplicated(
    [
        "title_key",
        "artist_key",
    ]
).any()

print(
    "Canonical Spotify title-artist rows:",
    len(spotify_exact_canonical)
)


Canonical Spotify title-artist rows: 469176


In [30]:
display(
    spotify_exact_canonical.loc[
        (
            (spotify_exact_canonical["title_key"] == "drive")
            & (spotify_exact_canonical["artist_key"] == "the cars")
        )
        |
        (
            (spotify_exact_canonical["title_key"] == "last last")
            & (spotify_exact_canonical["artist_key"] == "burna boy")
        ),
        [
            "title_key",
            "artist_key",
            "name",
            "primary_artist",
            "album_name",
            "popularity",
            "candidate_count",
            "signature_count",
            "signature_share",
            "id",
        ]
    ]
)


,title_key,artist_key,name,primary_artist,album_name,popularity,candidate_count,signature_count,signature_share,id
108996,drive,the cars,Drive,The Cars,Heartbeat City,68,70.0,37.0,0.528571,3wfujdbamR3Z46F4xav7LM
221706,last last,burna boy,Last Last,Burna Boy,"Love, Damini",68,22.0,21.0,0.954545,5YbPxJwPfrj7uswNwoF1pJ


### 13.4 Pass 1 — strict exact title + artist

This is the highest-confidence match.

The last observed run before this cleanup produced **10,466 strict exact matches out of 32,592 unique Billboard song-artist pairs**. Re-run the cell below so the result is generated from the current notebook state.


In [31]:
exact_matches = (
    billboard_songs
    .merge(
        spotify_exact_canonical,
        on=[
            "title_key",
            "artist_key",
        ],
        how="left",
        indicator="_merge_exact",
        suffixes=(
            "_billboard",
            "_spotify",
        ),
        validate="many_to_one",
    )
)

exact_match_map = (
    exact_matches.loc[
        exact_matches["_merge_exact"] == "both",
        [
            "Song",
            "Artist",
            "id",
        ]
    ]
    .rename(columns={"id": "spotify_id"})
    .copy()
)

exact_match_map["match_method"] = (
    "exact_title_artist"
)

print(
    "Strict exact matches:",
    len(exact_match_map)
)

print(
    f"Strict exact match rate: "
    f"{len(exact_match_map) / len(billboard_songs):.1%}"
)


Strict exact matches: 10466
Strict exact match rate: 32.1%


### 13.5 Pass 2 — exact title + lead artist after `Featuring`

A large share of Billboard credits put collaborators into one artist string, for example:

`Drake Featuring Jhené Aiko`

while the Spotify-derived table may use only `Drake` as the primary artist.

This pass is intentionally conservative:

- the title must still match **exactly** after normalization,
- only an explicit `Featuring ...` suffix is removed,
- `&` / `and` inside the lead act is preserved,
- `With`, aliases, and fuzzy matching are **not** handled yet.

The earlier single-title diagnostic recovered 328 clean examples, but that test covered only titles with one Spotify candidate. The production pass below applies the same rule to **all unmatched Billboard rows**.


In [32]:
def remove_featuring(artist_key):
    return re.sub(
        r"\s+featuring\s+.*$",
        "",
        artist_key,
    ).strip()


unmatched_after_exact = (
    exact_matches.loc[
        exact_matches["_merge_exact"] == "left_only",
        [
            "Song",
            "Artist",
            "title_key",
            "artist_key",
        ]
    ]
    .copy()
)

unmatched_after_exact["has_featuring_credit"] = (
    unmatched_after_exact["artist_key"]
    .str.contains(
        r"\bfeaturing\b",
        regex=True,
        na=False,
    )
)

unmatched_after_exact["billboard_lead_artist_key"] = (
    unmatched_after_exact["artist_key"]
    .map(remove_featuring)
)


In [33]:
featuring_candidates = (
    unmatched_after_exact.loc[
        unmatched_after_exact["has_featuring_credit"]
    ]
    .merge(
        spotify_exact_canonical,
        left_on=[
            "title_key",
            "billboard_lead_artist_key",
        ],
        right_on=[
            "title_key",
            "artist_key",
        ],
        how="left",
        suffixes=(
            "_billboard",
            "_spotify",
        ),
        validate="many_to_one",
    )
)

featuring_match_map = (
    featuring_candidates.loc[
        featuring_candidates["id"].notna(),
        [
            "Song",
            "Artist",
            "id",
        ]
    ]
    .rename(columns={"id": "spotify_id"})
    .copy()
)

featuring_match_map["match_method"] = (
    "exact_title_lead_artist_after_featuring"
)

print(
    "Pass-2 featuring matches:",
    len(featuring_match_map)
)


Pass-2 featuring matches: 798


### 13.6 Combine accepted deterministic matches

Each Billboard `Song` + `Artist` pair should appear at most once in the accepted map.

The combined table is intentionally just a **mapping layer**. Spotify features remain in the canonical lookup and can be merged back by `spotify_id` after additional matching passes are finished.


In [34]:
billboard_song_match_map = (
    pd.concat(
        [
            exact_match_map,
            featuring_match_map,
        ],
        ignore_index=True,
    )
)

duplicate_billboard_matches = (
    billboard_song_match_map
    .duplicated(
        [
            "Song",
            "Artist",
        ],
        keep=False,
    )
)

assert not duplicate_billboard_matches.any(), (
    "A Billboard song-artist pair was accepted by "
    "more than one deterministic pass."
)

billboard_song_matches = (
    billboard_songs
    .merge(
        billboard_song_match_map,
        on=[
            "Song",
            "Artist",
        ],
        how="left",
        validate="one_to_one",
    )
)

print(
    billboard_song_matches["match_method"]
    .value_counts(dropna=False)
)

print(
    f"Deterministic coverage so far: "
    f"{billboard_song_matches['spotify_id'].notna().mean():.1%}"
)


match_method
NaN                                        21328
exact_title_artist                         10466
exact_title_lead_artist_after_featuring      798
Name: count, dtype: int64
Deterministic coverage so far: 34.6%


### 13.7 Diagnose what remains unmatched

Do **not** fuzzy-match everything yet.

First separate remaining Billboard rows into:

- titles that exist somewhere in the canonical Spotify table but still have an artist mismatch,
- titles that do not have an exact normalized-title match at all.

This tells us whether the next deterministic rule should target artist formatting or title formatting.


In [35]:
remaining_unmatched = (
    billboard_song_matches.loc[
        billboard_song_matches["spotify_id"].isna(),
        [
            "Song",
            "Artist",
            "title_key",
            "artist_key",
        ]
    ]
    .copy()
)

spotify_title_counts = (
    spotify_exact_canonical
    .groupby("title_key")
    .size()
    .rename("spotify_title_candidates")
)

remaining_unmatched["spotify_title_candidates"] = (
    remaining_unmatched["title_key"]
    .map(spotify_title_counts)
    .fillna(0)
    .astype(int)
)

remaining_unmatched["title_exists_in_spotify"] = (
    remaining_unmatched["spotify_title_candidates"]
    .gt(0)
)

print(
    "Remaining unmatched:",
    len(remaining_unmatched)
)

print(
    "Exact title exists, artist still differs:",
    remaining_unmatched["title_exists_in_spotify"].sum()
)

print(
    "No exact Spotify title:",
    (~remaining_unmatched["title_exists_in_spotify"]).sum()
)


Remaining unmatched: 21328
Exact title exists, artist still differs: 11070
No exact Spotify title: 10258


In [36]:
display(
    remaining_unmatched.loc[
        remaining_unmatched["title_exists_in_spotify"]
    ]
    .sample(
        min(
            30,
            remaining_unmatched["title_exists_in_spotify"].sum(),
        ),
        random_state=42,
    )
)

display(
    remaining_unmatched.loc[
        ~remaining_unmatched["title_exists_in_spotify"]
    ]
    .sample(
        min(
            30,
            (~remaining_unmatched["title_exists_in_spotify"]).sum(),
        ),
        random_state=42,
    )
)


,Song,Artist,title_key,artist_key,spotify_title_candidates,title_exists_in_spotify
16441,You Don't Know,Scarlett & Black,you don t know,scarlett and black,22,True
20511,Heartbreaker,Mariah Carey Featuring Jay-Z,heartbreaker,mariah carey featuring jay z,32,True
25524,Why,Michelle Chamuel,why,michelle chamuel,90,True
21913,whoknows,Musiq,whoknows,musiq,1,True
23748,Every Girl,Young Money,every girl,young money,3,True
7220,The Way It Used To Be,Engelbert Humperdinck,the way it used to be,engelbert humperdinck,2,True
19495,Po Pimp,Do Or Die (Featuring Twista),po pimp,do or die featuring twista,1,True
3679,Ebb Tide,Lenny Welch,ebb tide,lenny welch,5,True
30761,Sirens,Travis Scott,sirens,travis scott,29,True
14099,Be Mine Tonight,Neil Diamond,be mine tonight,neil diamond,5,True


,Song,Artist,title_key,artist_key,spotify_title_candidates,title_exists_in_spotify
17564,Here Comes The Hammer,M.C. Hammer,here comes the hammer,m c hammer,0,False
18101,Take This Heart,Richard Marx,take this heart,richard marx,0,False
27503,Chun-Li,Nicki Minaj,chun li,nicki minaj,0,False
9695,Yesterday I Had The Blues,Harold Melvin And The Blue Notes,yesterday i had the blues,harold melvin and the blue notes,0,False
8424,Revival (Love Is Everywhere),The Allman Brothers Band,revival love is everywhere,the allman brothers band,0,False
6550,You've Got To Be Loved,The Montanas,you ve got to be loved,the montanas,0,False
32357,Earrings,Malcolm Todd,earrings,malcolm todd,0,False
21061,I Wanna Be Bad,Willa Ford,i wanna be bad,willa ford,0,False
13259,Walks Like A Lady,Journey,walks like a lady,journey,0,False
27286,Mayores,Becky G Featuring Bad Bunny,mayores,becky g featuring bad bunny,0,False


In [46]:
remaining_same_title = (
    remaining_unmatched.loc[
        remaining_unmatched["title_exists_in_spotify"]
    ]
    .copy()
)

remaining_same_title["billboard_lead_artist_key"] = (
    remaining_same_title["artist_key"]
    .map(remove_featuring)
)

spotify_artists_by_title = (
    spotify_exact_canonical
    .groupby("title_key")["artist_key"]
    .agg(set)
)

remaining_same_title["spotify_artist_candidates"] = (
    remaining_same_title["title_key"]
    .map(spotify_artists_by_title)
)

remaining_same_title["lead_artist_exists_for_title"] = (
    remaining_same_title.apply(
        lambda row:
        row["billboard_lead_artist_key"]
        in row["spotify_artist_candidates"],
        axis=1,
    )
)

In [47]:
print(
    remaining_same_title[
        "lead_artist_exists_for_title"
    ].value_counts()
)

lead_artist_exists_for_title
False    11070
Name: count, dtype: int64


In [48]:
remaining_featuring = (
    remaining_same_title.loc[
        remaining_same_title["artist_key"]
        .str.contains(
            r"\bfeaturing\b",
            regex=True,
            na=False,
        )
    ]
)

display(
    remaining_featuring[
        [
            "Song",
            "Artist",
            "billboard_lead_artist_key",
            "spotify_artist_candidates",
            "lead_artist_exists_for_title",
        ]
    ].head(30)
)

,Song,Artist,billboard_lead_artist_key,spotify_artist_candidates,lead_artist_exists_for_title
1253,Red Sails In The Sunset,The Platters Featuring Tony Williams,the platters,"{fats domino, guy lombardo and his royal canad...",False
1442,Send Me The Pillow You Dream On,The Browns Featuring Jim Edward Brown,the browns,"{johnny tillotson, dean martin, dolly parton, ...",False
1498,Blue Christmas,The Browns Featuring Jim Edward Brown,the browns,"{bright eyes, elvis presley, dean martin, cory...",False
1717,Ground Hog,The Browns Featuring Jim Edward Brown,the browns,"{spirit, pete seeger}",False
2063,Laugh,The Velvets featuring Virgil Johnson,the velvets,"{swans, sarcastic sounds}",False
2394,I'll Take You Home,"The Corsairs Featuring Jay ""Bird"" Uzzell",the corsairs,{the drifters},False
3010,Gone With The Wind,The Duprees featuring Joey Vann,the duprees,"{architects, chris connor, saint privat, ella ...",False
3447,She's Got Everything,The Essex Featuring Anita Humes,the essex,"{the romantics, krokus}",False
3722,Ronnie,"The 4 Seasons Featuring the ""Sound of Frankie ...",the 4 seasons,"{the rubinoos, frankie valli and the four seas...",False
3854,Rag Doll,"The 4 Seasons Featuring the ""Sound of Frankie ...",the 4 seasons,"{aerosmith, frankie valli and the four seasons}",False


In [49]:
print(
    remaining_featuring[
        "lead_artist_exists_for_title"
    ]
    .value_counts(dropna=False)
)

lead_artist_exists_for_title
False    1131
Name: count, dtype: int64


In [50]:
alias_review = (
    remaining_same_title
    .assign(
        billboard_artist_for_alias=(
            remaining_same_title["artist_key"]
            .map(remove_featuring)
        )
    )
    .groupby(
        "billboard_artist_for_alias"
    )
    .agg(
        unresolved_songs=("Song", "size"),
        example_titles=(
            "Song",
            lambda x: " | ".join(
                x.astype(str).head(5)
            ),
        ),
    )
    .sort_values(
        "unresolved_songs",
        ascending=False,
    )
)

display(
    alias_review.head(30)
)

,unresolved_songs,example_titles
billboard_artist_for_alias,,
taylor swift,94,Our Song | Change | Love Story | Fearless | Yo...
glee cast,87,Don't Stop Believin' | Rehab | Alone | Maybe T...
justin bieber,62,One Time | Love Me | Favorite Girl | Down To E...
post malone,47,Deja Vu | Congratulations | Rockstar | I Fall ...
beyonce,47,"Crazy In Love | Baby Boy | ""Me, Myself And I"" ..."
elvis presley with the jordanaires,46,Hard Headed Woman | Don't Ask Me Why | I Need ...
rod wave,43,Heart On Ice | Fuck The World | Thief In The N...
madonna,42,Holiday | Borderline | Lucky Star | Like A Vir...
the weeknd,42,Often | Earned It (Fifty Shades Of Grey) | Whe...


In [51]:
taylor_unresolved = (
    remaining_same_title.loc[
        remaining_same_title[
            "billboard_lead_artist_key"
        ] == "taylor swift"
    ]
    .copy()
)

print(
    "Unresolved Taylor Swift songs:",
    len(taylor_unresolved)
)

display(
    taylor_unresolved[
        [
            "Song",
            "Artist",
            "title_key",
            "spotify_title_candidates",
        ]
    ].head(20)
)

Unresolved Taylor Swift songs: 94


,Song,Artist,title_key,spotify_title_candidates
23122,Our Song,Taylor Swift,our song,13
23446,Change,Taylor Swift,change,67
23477,Love Story,Taylor Swift,love story,16
23520,Fearless,Taylor Swift,fearless,34
23541,You're Not Sorry,Taylor Swift,you re not sorry,1
23549,You Belong With Me,Taylor Swift,you belong with me,2
23557,White Horse,Taylor Swift,white horse,2
23561,Forever & Always,Taylor Swift,forever and always,4
23563,The Way I Loved You,Taylor Swift,the way i loved you,1
23564,Fifteen,Taylor Swift,fifteen,2


In [52]:
taylor_candidate_check = (
    taylor_unresolved[
        [
            "Song",
            "title_key",
        ]
    ]
    .head(10)
    .merge(
        spotify_exact_canonical[
            [
                "title_key",
                "name",
                "primary_artist",
                "artist_key",
                "album_name",
            ]
        ],
        on="title_key",
        how="left",
    )
)

display(
    taylor_candidate_check[
        [
            "Song",
            "name",
            "primary_artist",
            "album_name",
        ]
    ]
)

,Song,name,primary_artist,album_name
0,Our Song,Our Song,Fareoh,Our Song
1,Our Song,Our Song,Kenichiro Nishihara,Elastic Afterwords
2,Our Song,Our Song,Lil’ Ashes,Be Little
3,Our Song,Our Song,Matchbox Twenty,North (Deluxe Edition)
4,Our Song,Our Song,Morgan Page,Our Song (Mix)
...,...,...,...,...
137,Forever & Always,Forever (And Always),Lefty Frizzell,"The Complete Columbia Recording Sessions, Vol...."
138,Forever & Always,Forever And Always,M.O.P.,Foundation
139,The Way I Loved You,The Way I Loved You,Tyler Ward,"Tyler Ward Covers, Vol. 2"
140,Fifteen,Fifteen,Aiden,Our Gangs Dark Oath


In [53]:
taylor_spotify = (
    songs.loc[
        songs["artist_key"] == "taylor swift",
        [
            "name",
            "album_name",
            "title_key",
            "year",
            "popularity",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "album_name",
            "name",
        ]
    )
)

print(
    "Taylor Swift rows in 550K:",
    len(taylor_spotify)
)

display(
    taylor_spotify.head(100)
)

Taylor Swift rows in 550K: 34


,name,album_name,title_key,year,popularity
2440,...Ready For It? - BloodPop® Remix,...Ready For It? (BloodPop® Remix),ready for it bloodpop remix,2017,52
25745,All You Had To Do Was Stay,1989,all you had to do was stay,2014,57
25746,All You Had To Do Was Stay,1989 (Deluxe Edition),all you had to do was stay,2014,47
4971,22,22,22,2012,0
24175,All Of The Girls You Loved Before,All Of The Girls You Loved Before,all of the girls you loved before,2019,71
25523,All Too Well (Sad Girl Autumn Version) - Recor...,All Too Well (Sad Girl Autumn Version) - Recor...,all too well sad girl autumn version recorded ...,2021,63
33393,Anti-Hero - ILLENIUM Remix,Anti-Hero (ILLENIUM Remix),anti hero illenium remix,2022,51
43733,Bad Blood,Bad Blood,bad blood,2015,66
446438,That’s When (feat. Keith Urban) (Taylor’s Vers...,Fearless (Taylor's Version),that s when feat keith urban taylor s version ...,2021,59
537114,You All Over Me (feat. Maren Morris) (Taylor’s...,Fearless (Taylor's Version),you all over me feat maren morris taylor s ver...,2021,57


In [54]:
def find_taylor_title_variants(title_key):
    matches = (
        taylor_spotify.loc[
            taylor_spotify["title_key"]
            .str.startswith(title_key)
        ]
    )

    return matches["name"].tolist()


taylor_unresolved["possible_title_variants"] = (
    taylor_unresolved["title_key"]
    .apply(find_taylor_title_variants)
)

display(
    taylor_unresolved.loc[
        taylor_unresolved[
            "possible_title_variants"
        ].str.len() > 0,
        [
            "Song",
            "Artist",
            "possible_title_variants",
        ]
    ].head(30)
)

,Song,Artist,possible_title_variants
26047,Wildest Dreams,Taylor Swift,[Wildest Dreams - R3hab Remix]
28108,ME!,Taylor Swift Featuring Brendon Urie,[ME! (feat. Brendon Urie of Panic! At The Disco)]
29116,Coney Island,Taylor Swift Featuring The National,[coney island (feat. The National)]
30307,Anti-Hero,Taylor Swift,[Anti-Hero - ILLENIUM Remix]


In [55]:
def remove_title_feature_credit(title_key):
    return re.sub(
        r"\s+(?:feat|featuring)\s+.*$",
        "",
        title_key
    ).strip()

In [56]:
spotify_exact_canonical["title_no_feature_key"] = (
    spotify_exact_canonical["title_key"]
    .map(remove_title_feature_credit)
)

In [57]:
title_feature_candidates = (
    remaining_unmatched
    .assign(
        billboard_lead_artist_key=(
            remaining_unmatched["artist_key"]
            .map(remove_featuring)
        )
    )
    .merge(
        spotify_exact_canonical,
        left_on=[
            "title_key",
            "billboard_lead_artist_key",
        ],
        right_on=[
            "title_no_feature_key",
            "artist_key",
        ],
        how="inner",
        suffixes=(
            "_billboard",
            "_spotify",
        ),
    )
)

print(
    "Possible title-feature-credit matches:",
    len(title_feature_candidates)
)

Possible title-feature-credit matches: 1097


In [58]:
display(
    title_feature_candidates[
        [
            "Song",
            "Artist",
            "name",
            "primary_artist",
            "title_key_billboard",
            "title_key_spotify",
        ]
    ].sample(
        min(30, len(title_feature_candidates)),
        random_state=42,
    )
)

,Song,Artist,name,primary_artist,title_key_billboard,title_key_spotify
44,Someone,SWV (Featuring Puff Daddy),Someone (feat. Puff Daddy),SWV,someone,someone feat puff daddy
568,Roses,The Chainsmokers Featuring ROZES,Roses (feat. ROZES) - Loosid Remix,The Chainsmokers,roses,roses feat rozes loosid remix
56,So Good,Davina,So Good (feat. Raekwon),Davina,so good,so good feat raekwon
636,Party,Chris Brown Featuring Usher & Gucci Mane,Party (feat. Usher & Gucci Mane),Chris Brown,party,party feat usher and gucci mane
486,Old School Love,Lupe Fiasco Featuring Ed Sheeran,Old School Love (feat. Ed Sheeran),Lupe Fiasco,old school love,old school love feat ed sheeran
96,Shut Up,"""Trick Daddy Featuring Duece Poppito, Trina, Co""","Shut Up (feat. Duece Poppito, Trina, & Co)",Trick Daddy,shut up,shut up feat duece poppito trina and co
761,Best Part,Daniel Caesar Featuring H.E.R.,Best Part (feat. H.E.R.),Daniel Caesar,best part,best part feat h e r
51,All Cried Out,Allure Featuring 112,All Cried Out (feat. 112) - Behind the scenes ...,Allure,all cried out,all cried out feat 112 behind the scenes video
107,Chickenhead,Project Pat,Chickenhead (feat. La Chat & Three 6 Mafia),Project Pat,chickenhead,chickenhead feat la chat and three 6 mafia
666,Loyalty.,Kendrick Lamar Featuring Rihanna,LOYALTY. FEAT. RIHANNA.,Kendrick Lamar,loyalty,loyalty feat rihanna


In [59]:
def remove_spotify_title_feature_credit(title):
    if pd.isna(title):
        return ""

    title = str(title)

    # Remove a parenthetical feature credit.
    # Anything AFTER the parenthesis remains.
    title = re.sub(
        r"\s*[\(\[]\s*(?:feat\.?|featuring)\s+[^\)\]]+[\)\]]",
        "",
        title,
        flags=re.IGNORECASE,
    )

    # Remove a terminal "feat. Artist" clause only when it
    # actually ends the title.
    title = re.sub(
        r"\s+(?:feat\.?|featuring)\s+[^-]+$",
        "",
        title,
        flags=re.IGNORECASE,
    )

    return title.strip()

In [60]:
spotify_exact_canonical["title_no_feature_key"] = (
    spotify_exact_canonical["name"]
    .map(remove_spotify_title_feature_credit)
    .map(normalize_text)
)

In [61]:
title_feature_candidates = (
    remaining_unmatched
    .assign(
        billboard_lead_artist_key=(
            remaining_unmatched["artist_key"]
            .map(remove_featuring)
        )
    )
    .merge(
        spotify_exact_canonical,
        left_on=[
            "title_key",
            "billboard_lead_artist_key",
        ],
        right_on=[
            "title_no_feature_key",
            "artist_key",
        ],
        how="inner",
        suffixes=(
            "_billboard",
            "_spotify",
        ),
    )
)

print(
    "Safer title-feature candidates:",
    len(title_feature_candidates)
)

Safer title-feature candidates: 815


In [62]:
display(
    title_feature_candidates[
        [
            "Song",
            "Artist",
            "name",
            "primary_artist",
            "title_key_billboard",
            "title_no_feature_key",
        ]
    ].sample(
        min(30, len(title_feature_candidates)),
        random_state=42,
    )
)

,Song,Artist,name,primary_artist,title_key_billboard,title_no_feature_key
196,"""Want It, Need It""",Plies Featuring Ashanti,"Want It, Need It (feat. Ashanti)",Plies,want it need it,want it need it
299,My Moment,"""DJ Drama Featuring 2 Chainz, Meek Mill & Jere...","My Moment (feat. 2 Chainz, Meek Mill and Jeremih)",DJ Drama,my moment,my moment
388,Work From Home,Fifth Harmony Featuring Ty Dolla $ign,Work from Home (feat. Ty Dolla $ign),Fifth Harmony,work from home,work from home
227,Club Can't Handle Me,Flo Rida Featuring David Guetta,Club Can't Handle Me (feat. David Guetta),Flo Rida,club can t handle me,club can t handle me
664,Demons,Drake Featuring Fivio Foreign & Sosa Geek,Demons (feat. Fivio Foreign & Sosa Geek),Drake,demons,demons
801,Forever,Lil Baby Featuring Fridayy,Forever (feat. Lil Wayne),Lil Baby,forever,forever
199,When Love Takes Over,David Guetta Featuring Kelly Rowland,When Love Takes Over (feat. Kelly Rowland),David Guetta,when love takes over,when love takes over
336,No Mediocre,T.I. Featuring Iggy Azalea,No Mediocre (feat. Iggy Azalea),T.I.,no mediocre,no mediocre
338,About The Money,T.I. Featuring Young Thug,About the Money (feat. Young Thug),T.I.,about the money,about the money
528,Dark Side Of The Moon,Lil Wayne Featuring Nicki Minaj,Dark Side Of The Moon (feat. Nicki Minaj),Lil Wayne,dark side of the moon,dark side of the moon


In [63]:
title_feature_candidates["spotify_artist_list"] = (
    title_feature_candidates["artists"]
    .apply(parse_artist_list)
)

title_feature_candidates["spotify_artist_keys"] = (
    title_feature_candidates["spotify_artist_list"]
    .apply(
        lambda artists: [
            normalize_text(artist)
            for artist in artists
        ]
    )
)

title_feature_candidates["spotify_secondary_keys"] = (
    title_feature_candidates["spotify_artist_keys"]
    .apply(lambda artists: artists[1:])
)

In [64]:
title_feature_candidates["billboard_has_featuring"] = (
    title_feature_candidates["artist_key_billboard"]
    .str.contains(
        r"\bfeaturing\b",
        regex=True,
        na=False,
    )
)

In [65]:
def has_secondary_artist_overlap(row):
    return any(
        artist
        and artist in row["artist_key_billboard"]
        for artist in row["spotify_secondary_keys"]
    )


title_feature_candidates["secondary_artist_overlap"] = (
    title_feature_candidates
    .apply(
        has_secondary_artist_overlap,
        axis=1,
    )
)

In [66]:
pd.crosstab(
    title_feature_candidates[
        "billboard_has_featuring"
    ],
    title_feature_candidates[
        "secondary_artist_overlap"
    ],
)

secondary_artist_overlap,False,True
billboard_has_featuring,,
False,55,0
True,32,728


In [67]:
strong_title_feature_candidates = (
    title_feature_candidates.loc[
        title_feature_candidates["billboard_has_featuring"]
        & title_feature_candidates["secondary_artist_overlap"]
    ]
    .copy()
)

candidate_counts = (
    strong_title_feature_candidates
    .groupby(["Song", "Artist"])
    .size()
    .value_counts()
    .sort_index()
)

print(candidate_counts)

1    700
2     14
Name: count, dtype: int64


In [68]:
print(
    "Candidate rows:",
    len(strong_title_feature_candidates)
)

print(
    "Unique Billboard songs:",
    strong_title_feature_candidates[
        ["Song", "Artist"]
    ].drop_duplicates().shape[0]
)

Candidate rows: 728
Unique Billboard songs: 714


In [69]:
ambiguous_title_feature_songs = (
    strong_title_feature_candidates
    .groupby(["Song", "Artist"])
    .filter(lambda group: len(group) > 1)
)

display(
    ambiguous_title_feature_songs[
        [
            "Song",
            "Artist",
            "name",
            "primary_artist",
            "artists",
            "album_name",
            "popularity",
            "id",
        ]
    ]
    .sort_values(
        ["Song", "Artist", "popularity"],
        ascending=[True, True, False],
    )
)

,Song,Artist,name,primary_artist,artists,album_name,popularity,id
380,Bang My Head,David Guetta Featuring Sia & Fetty Wap,Bang My Head (feat. Sia & Fetty Wap),David Guetta,"[""David Guetta"", ""Fetty Wap"", ""Sia""]",Listen Again,59,53Y0kdCa1CZ9gRqEuknfwy
379,Bang My Head,David Guetta Featuring Sia & Fetty Wap,Bang My Head (feat. Sia),David Guetta,"[""David Guetta"", ""Sia""]",Listen (Deluxe),31,6z7HKLe33Nxca2w3WmXBoK
156,Come To Me,Diddy Featuring Nicole Scherzinger,Come to Me (feat. Nicole Scherzinger),Diddy,"[""Diddy"", ""Nicole Scherzinger""]",Press Play,52,6yuvC80FcnVJNvC0DbXN9e
157,Come To Me,Diddy Featuring Nicole Scherzinger,"Come to Me (feat. Nicole Scherzinger, Yung Joc...",Diddy,"[""Diddy"", ""Nicole Scherzinger"", ""Yung Joc"", ""Y...",Tell Me,22,1unO85fSNONHm2zklxkolp
735,Essence,Wizkid Featuring Justin Bieber & Tems,Essence (feat. Tems),Wizkid,"[""Wizkid"", ""Tems""]",Made In Lagos,75,5FG7Tl93LdH117jEKYl3Cm
734,Essence,Wizkid Featuring Justin Bieber & Tems,Essence (feat. Justin Bieber & Tems),Wizkid,"[""Wizkid"", ""Justin Bieber"", ""Tems""]",Made In Lagos: Deluxe Edition,58,1MZtr7IH5qtjIkqrXj8WOJ
286,Ima Boss,"""Meek Mill Featuring T.I., Birdman, Lil Wayne,...",Ima Boss (feat. Rick Ross),Meek Mill,"[""Meek Mill"", ""Rick Ross""]","MMG Presents: Self Made, Vol. 1",67,7sjwaBQ6M3aVigYfBjm1sL
287,Ima Boss,"""Meek Mill Featuring T.I., Birdman, Lil Wayne,...","Ima Boss (feat. T.I., Birdman, Lil' Wayne, DJ ...",Meek Mill,"[""Meek Mill"", ""Birdman"", ""DJ Khaled"", ""Lil Way...",Ima Boss (Remix Version),49,2npw9XuCQpHXM9ramYL4HR
100,Into You,Fabolous Featuring Tamia Or Ashanti,Into You (feat. Tamia),Fabolous,"[""Fabolous"", ""Tamia""]",Street Dreams (Bonus Track),63,0yHConG49418wPs8UfAt44
99,Into You,Fabolous Featuring Tamia Or Ashanti,Into You (feat. Ashanti),Fabolous,"[""Fabolous"", ""Ashanti""]",Street Dreams (Bonus Track),54,4b5aJdOgxps6z0dPe6wCHF


In [70]:
pass3_unique_matches = (
    strong_title_feature_candidates
    .groupby(["Song", "Artist"])
    .filter(lambda group: len(group) == 1)
    .copy()
)

pass3_unique_match_map = (
    pass3_unique_matches[
        [
            "Song",
            "Artist",
            "id",
        ]
    ]
    .rename(columns={"id": "spotify_id"})
)

pass3_unique_match_map["match_method"] = (
    "exact_title_after_feature_strip_"
    "lead_artist_secondary_overlap"
)

print(
    "Accepted Pass-3 unique matches:",
    len(pass3_unique_match_map)
)

Accepted Pass-3 unique matches: 700


In [71]:
duplicate_version_titles = [
    "Let's Go",
    "Neck & Wrist",
    "Off The Books",
    "Shut Up",
    "Writing On The Wall",
]

display(
    ambiguous_title_feature_songs.loc[
        ambiguous_title_feature_songs["Song"]
        .isin(duplicate_version_titles),
        [
            "Song",
            "name",
            "artists",
            "album_name",
            "popularity",
            "danceability",
            "energy",
            "valence",
            "tempo",
            "duration_ms",
            "signature_count",
            "id",
        ]
    ]
    .sort_values(
        ["Song", "popularity"],
        ascending=[True, False],
    )
)

,Song,name,artists,album_name,popularity,danceability,energy,valence,tempo,duration_ms,signature_count,id
123,Let's Go,Let's Go (feat. Twista & Lil' Jon),"[""Trick Daddy"", ""Lil Jon"", ""Twista""]",Thug Matrimony: Married to the Streets,66,0.748,0.767,0.519,133.929,222200,1.0,1fLibtVj3MrNTwRZq5yUOs
122,Let's Go,Let's Go (feat. Big D & Twista),"[""Trick Daddy"", ""Lil Jon"", ""Twista""]",Thug Matrimony: Married to the Streets,47,0.754,0.711,0.579,134.059,223627,1.0,4MatugKuTLRqQGLss6Ch4a
776,Neck & Wrist,Neck & Wrist (feat. JAY-Z & Pharrell Williams),"[""Pusha T"", ""JAY-Z"", ""Pharrell Williams""]",It's Almost Dry,51,0.625,0.542,0.328,152.403,209315,1.0,7kQJCw0ZkvHgfJqRwPblmG
775,Neck & Wrist,Neck & Wrist (feat. JAY-Z & Pharrell),"[""Pusha T"", ""JAY-Z"", ""Pharrell Williams""]",Neck & Wrist,41,0.625,0.542,0.328,152.403,209315,1.0,5QNW6Vx14N4OfJ7D0eDg0d
29,Off The Books,Off the Books (feat. Big Pun & Cuban Link),"[""The Beatnuts"", ""Big Pun"", ""Cuban Link""]",Endangered Species,50,0.836,0.778,0.760,100.512,168773,1.0,5qqBsCBwzv0g0xb1kB1TGJ
30,Off The Books,Off the Books (feat. Big Punisher & Cuban Linx...,"[""The Beatnuts"", ""Big Pun"", ""Cuban Link""]",The Legacy: The Best Of Big Pun,35,0.840,0.797,0.743,100.497,167520,1.0,3zoExZ6fKHXOUbZfS0zT3C
58,Shut Up,"Shut Up (feat. Duece Poppito of 24 Karatz, Tri...","[""Trick Daddy"", ""Duece Poppito"", ""Trina"", ""Co ...",Book of Thugs: Chapter AK Verse 47,47,0.792,0.733,0.653,144.857,262787,1.0,66MsAs4nVj68t6va1lygIm
59,Shut Up,"Shut Up (feat. Duece Poppito, Trina, & Co)","[""Trick Daddy"", ""Duece Poppito"", ""Trina"", ""Co""]",Book of Thugs: Chapter AK Verse 47,21,0.806,0.738,0.640,144.850,261840,1.0,1a6Wu2fORbFmZbAQryjr4u
614,Writing On The Wall,Writing on the Wall (feat. Post Malone & Cardi B),"[""French Montana"", ""Rvssian"", ""Post Malone"", ""...",Writing on the Wall (feat. Post Malone & Cardi B),51,0.773,0.836,0.497,112.010,201271,1.0,7x9nXsowok1JszkVztI5NI
615,Writing On The Wall,"Writing on the Wall (feat. Post Malone, Cardi ...","[""French Montana"", ""Post Malone"", ""Cardi B"", ""...",MONTANA,47,0.774,0.836,0.494,112.014,201271,1.0,7ntUbTIwmwOD6ygCMRZT6e


In [72]:
manual_ambiguous_overrides = {
    (
        "Let's Go",
        "Trick Daddy Featuring Lil Jon & Twista",
    ): "1fLibtVj3MrNTwRZq5yUOs",

    (
        "Neck & Wrist",
        "Pusha T Featuring Jay-Z & Pharrell Williams",
    ): "7kQJCw0ZkvHgfJqRwPblmG",

    (
        "Off The Books",
        "The Beatnuts Featuring Big Punisher & Cuban Link",
    ): "5qqBsCBwzv0g0xb1kB1TGJ",

    (
        "Shut Up",
        "Trick Daddy Featuring Duece Poppito, Trina, Co",
    ): "66MsAs4nVj68t6va1lygIm",

    (
        "Writing On The Wall",
        "French Montana Featuring Post Malone, Cardi B...",
    ): "7x9nXsowok1JszkVztI5NI",
}

In [73]:
pd.set_option(
    "display.max_colwidth",
    None,
)

display(
    ambiguous_title_feature_songs[
        [
            "Song",
            "Artist",
            "name",
            "artists",
            "id",
        ]
    ]
    .sort_values(
        ["Song", "Artist"]
    )
)

,Song,Artist,name,artists,id
379,Bang My Head,David Guetta Featuring Sia & Fetty Wap,Bang My Head (feat. Sia),"[""David Guetta"", ""Sia""]",6z7HKLe33Nxca2w3WmXBoK
380,Bang My Head,David Guetta Featuring Sia & Fetty Wap,Bang My Head (feat. Sia & Fetty Wap),"[""David Guetta"", ""Fetty Wap"", ""Sia""]",53Y0kdCa1CZ9gRqEuknfwy
156,Come To Me,Diddy Featuring Nicole Scherzinger,Come to Me (feat. Nicole Scherzinger),"[""Diddy"", ""Nicole Scherzinger""]",6yuvC80FcnVJNvC0DbXN9e
157,Come To Me,Diddy Featuring Nicole Scherzinger,"Come to Me (feat. Nicole Scherzinger, Yung Joc, Young Dro & T.I.)","[""Diddy"", ""Nicole Scherzinger"", ""Yung Joc"", ""Young Dro"", ""T.I.""]",1unO85fSNONHm2zklxkolp
734,Essence,Wizkid Featuring Justin Bieber & Tems,Essence (feat. Justin Bieber & Tems),"[""Wizkid"", ""Justin Bieber"", ""Tems""]",1MZtr7IH5qtjIkqrXj8WOJ
735,Essence,Wizkid Featuring Justin Bieber & Tems,Essence (feat. Tems),"[""Wizkid"", ""Tems""]",5FG7Tl93LdH117jEKYl3Cm
286,Ima Boss,"""Meek Mill Featuring T.I., Birdman, Lil Wayne, DJ Khaled, Rick Ross & Swizz Beatz""",Ima Boss (feat. Rick Ross),"[""Meek Mill"", ""Rick Ross""]",7sjwaBQ6M3aVigYfBjm1sL
287,Ima Boss,"""Meek Mill Featuring T.I., Birdman, Lil Wayne, DJ Khaled, Rick Ross & Swizz Beatz""","Ima Boss (feat. T.I., Birdman, Lil' Wayne, DJ Khaled, Rick Ross & Swizz Beatz)","[""Meek Mill"", ""Birdman"", ""DJ Khaled"", ""Lil Wayne"", ""Rick Ross"", ""Swizz Beatz"", ""T.I.""]",2npw9XuCQpHXM9ramYL4HR
99,Into You,Fabolous Featuring Tamia Or Ashanti,Into You (feat. Ashanti),"[""Fabolous"", ""Ashanti""]",4b5aJdOgxps6z0dPe6wCHF
100,Into You,Fabolous Featuring Tamia Or Ashanti,Into You (feat. Tamia),"[""Fabolous"", ""Tamia""]",0yHConG49418wPs8UfAt44


In [78]:
pass3_manual_ids = [
    # Billboard collaborator credit identifies the recording
    "53Y0kdCa1CZ9gRqEuknfwy",  # Bang My Head — Sia & Fetty Wap
    "6yuvC80FcnVJNvC0DbXN9e",  # Come To Me — Nicole Scherzinger
    "1MZtr7IH5qtjIkqrXj8WOJ",  # Essence — Justin Bieber & Tems
    "2npw9XuCQpHXM9ramYL4HR",  # Ima Boss — remix cast matches Billboard
    "50DMJJpAeQv4fIpxZvQz2e",  # Monsters — Demi Lovato & blackbear
    "31dgrKXIJ7FCxlENs8uxuo",  # Show Me Love — Miguel only
    "2eSJflipjhSKLExuSwuFrO",  # What's Luv? — Ashanti only

    # Duplicate / alternate-metadata representations reviewed manually
    "1fLibtVj3MrNTwRZq5yUOs",  # Let's Go
    "7kQJCw0ZkvHgfJqRwPblmG",  # Neck & Wrist
    "5qqBsCBwzv0g0xb1kB1TGJ",  # Off The Books
    "66MsAs4nVj68t6va1lygIm",  # Shut Up
    "7x9nXsowok1JszkVztI5NI",  # Writing On The Wall
]

In [79]:
pass3_manual_match_map = (
    ambiguous_title_feature_songs.loc[
        ambiguous_title_feature_songs["id"]
        .isin(pass3_manual_ids),
        [
            "Song",
            "Artist",
            "id",
        ]
    ]
    .rename(
        columns={
            "id": "spotify_id"
        }
    )
    .drop_duplicates()
    .reset_index(drop=True)
)

pass3_manual_match_map["match_method"] = (
    "manual_ambiguous_feature_credit_review"
)

In [80]:
print(
    "Manually resolved:",
    len(pass3_manual_match_map)
)

assert len(pass3_manual_match_map) == 12

assert not pass3_manual_match_map.duplicated(
    [
        "Song",
        "Artist",
    ]
).any()

display(
    pass3_manual_match_map
)

Manually resolved: 12


,Song,Artist,spotify_id,match_method
0,Off The Books,The Beatnuts Featuring Big Punisher & Cuban Link,5qqBsCBwzv0g0xb1kB1TGJ,manual_ambiguous_feature_credit_review
1,Shut Up,"""Trick Daddy Featuring Duece Poppito, Trina, Co""",66MsAs4nVj68t6va1lygIm,manual_ambiguous_feature_credit_review
2,What's Luv?,Fat Joe Featuring Ashanti,2eSJflipjhSKLExuSwuFrO,manual_ambiguous_feature_credit_review
3,Let's Go,Trick Daddy Featuring Lil Jon & Twista,1fLibtVj3MrNTwRZq5yUOs,manual_ambiguous_feature_credit_review
4,Come To Me,Diddy Featuring Nicole Scherzinger,6yuvC80FcnVJNvC0DbXN9e,manual_ambiguous_feature_credit_review
5,Ima Boss,"""Meek Mill Featuring T.I., Birdman, Lil Wayne, DJ Khaled, Rick Ross & Swizz Beatz""",2npw9XuCQpHXM9ramYL4HR,manual_ambiguous_feature_credit_review
6,Bang My Head,David Guetta Featuring Sia & Fetty Wap,53Y0kdCa1CZ9gRqEuknfwy,manual_ambiguous_feature_credit_review
7,Writing On The Wall,"""French Montana Featuring Post Malone, Cardi B & Rvssian""",7x9nXsowok1JszkVztI5NI,manual_ambiguous_feature_credit_review
8,Show Me Love,Alicia Keys Featuring Miguel,31dgrKXIJ7FCxlENs8uxuo,manual_ambiguous_feature_credit_review
9,Monsters,All Time Low Featuring Demi Lovato & Blackbear,50DMJJpAeQv4fIpxZvQz2e,manual_ambiguous_feature_credit_review


In [81]:
pass3_match_map = (
    pd.concat(
        [
            pass3_unique_match_map,
            pass3_manual_match_map,
        ],
        ignore_index=True,
    )
)

print(
    "Total accepted Pass-3 matches:",
    len(pass3_match_map)
)

assert len(pass3_match_map) == 712

assert not pass3_match_map.duplicated(
    ["Song", "Artist"]
).any()

Total accepted Pass-3 matches: 712


In [82]:
billboard_song_match_map = (
    pd.concat(
        [
            exact_match_map,
            featuring_match_map,
            pass3_match_map,
        ],
        ignore_index=True,
    )
)

assert not billboard_song_match_map.duplicated(
    ["Song", "Artist"]
).any()

billboard_song_matches = (
    billboard_songs
    .merge(
        billboard_song_match_map,
        on=["Song", "Artist"],
        how="left",
        validate="one_to_one",
    )
)

print(
    billboard_song_matches["match_method"]
    .value_counts(dropna=False)
)

print(
    f"\nMatched songs: "
    f"{billboard_song_matches['spotify_id'].notna().sum():,}"
)

print(
    f"Coverage: "
    f"{billboard_song_matches['spotify_id'].notna().mean():.1%}"
)

match_method
NaN                                                              20616
exact_title_artist                                               10466
exact_title_lead_artist_after_featuring                            798
exact_title_after_feature_strip_lead_artist_secondary_overlap      700
manual_ambiguous_feature_credit_review                              12
Name: count, dtype: int64

Matched songs: 11,976
Coverage: 36.7%


In [83]:
billboard_weekly_matches = (
    billboard
    .merge(
        billboard_song_matches[
            [
                "Song",
                "Artist",
                "spotify_id",
                "match_method",
            ]
        ],
        on=[
            "Song",
            "Artist",
        ],
        how="left",
        validate="many_to_one",
    )
)

billboard_weekly_matches["year"] = (
    billboard_weekly_matches["Date"]
    .dt.year
)

In [84]:
yearly_match_coverage = (
    billboard_weekly_matches
    .groupby("year")
    .agg(
        weekly_rows=(
            "Song",
            "size",
        ),
        matched_weekly_rows=(
            "spotify_id",
            lambda x: x.notna().sum(),
        ),
    )
)

yearly_match_coverage["weekly_match_rate"] = (
    yearly_match_coverage["matched_weekly_rows"]
    / yearly_match_coverage["weekly_rows"]
)

display(
    yearly_match_coverage.tail(40)
)

,weekly_rows,matched_weekly_rows,weekly_match_rate
year,,,
1987,5200,2319,0.445962
1988,5200,2414,0.464231
1989,5199,2258,0.434314
1990,5200,2381,0.457885
1991,5200,2247,0.432115
1992,5300,2334,0.440377
1993,5200,2352,0.452308
1994,5200,2606,0.501154
1995,5200,2384,0.458462


In [85]:
ANALYSIS_START = "1999-01-01"
ANALYSIS_END = "2018-12-31"

billboard_analysis = (
    billboard_weekly_matches.loc[
        billboard_weekly_matches["Date"]
        .between(
            ANALYSIS_START,
            ANALYSIS_END,
        )
    ]
    .copy()
)

print(
    "Analysis date range:",
    billboard_analysis["Date"].min(),
    "to",
    billboard_analysis["Date"].max(),
)

print(
    "Weekly rows:",
    f"{len(billboard_analysis):,}",
)

print(
    "Matched weekly rows:",
    f"{billboard_analysis['spotify_id'].notna().sum():,}",
)

print(
    "Overall weekly coverage:",
    f"{billboard_analysis['spotify_id'].notna().mean():.1%}",
)

Analysis date range: 1999-01-06 00:00:00 to 2018-12-26 00:00:00
Weekly rows: 104,300
Matched weekly rows: 59,301
Overall weekly coverage: 56.9%


### Matching checkpoint

At this point, stop and inspect the new counts before adding another rule.

Likely later passes include:

- carefully handling `With` credits,
- known artist aliases / stage-name variants,
- title-version differences,
- fuzzy matching only within constrained candidate sets.

Do not use a title-only match by itself as evidence of identity: common titles such as `Stay`, `Monster`, or `Drive` can belong to many unrelated artists.


# Appendix — preserved thought process / retired exploration

These cells are intentionally commented out. They document how the current decisions were reached but are no longer required for the main pipeline.

In [37]:
# --- Historical matching exploration / validation ---
#
# The production matching pipeline above supersedes these one-off diagnostics.
#
# 1) Duplicate-resolution validation:
#
# drive_duplicates = songs.loc[
#     (songs["title_key"] == "drive")
#     & (songs["artist_key"] == "the cars")
# ]
#
# last_last_duplicates = songs.loc[
#     (songs["title_key"] == "last last")
#     & (songs["artist_key"] == "burna boy")
# ]
#
# These checks showed:
# - Drive — The Cars: 70 candidate rows, dominant profile count = 37.
# - Last Last — Burna Boy: 22 candidate rows, dominant profile count = 21.
#
# 2) Retired title-only diagnostic:
#
# A prior merge using only `title_key` expanded 22,126 unmatched Billboard rows
# into 122,498 candidate pairings. That number was NOT a song-level match count.
# The production notebook now uses title membership / candidate counts instead
# of interpreting a many-to-many title-only merge as coverage.
#
# 3) Retired single-title-only Featuring test:
#
# An earlier diagnostic limited the `Featuring` rule to exact titles with one
# Spotify candidate and found 328 clean matches. The production Pass 2 now
# applies the same conservative rule to all unmatched Billboard rows.


In [38]:
# --- Historical exploration: previewing the huge raw CSV ---
#
# song_preview = pd.read_csv(
#     SONGS_PATH,
#     nrows=5,
# )
#
# print("Columns:", song_preview.columns.tolist())
# display(song_preview)

In [39]:
# --- Historical exploration: suspicious release years ---
#
# songs["year"].value_counts().sort_index()
#
# print("Earliest year:", songs["year"].min())
# print("Latest year:", songs["year"].max())
#
# # This revealed obvious errors such as modern songs assigned to 1900.
# # Decision: do not use the 550K `year` field as the forecasting timeline.
#
# display(
#     songs.loc[
#         songs["year"] < 1920,
#         ["name", "artists", "year", "genre", "niche_genres"],
#     ]
# )

In [40]:
# --- Historical exploration: auditing the source Folk category ---
#
# display(
#     songs.loc[
#         songs["genre"] == "Folk",
#         ["name", "artists", "genre", "niche_genres"],
#     ]
# )
#
# # This is where systematic reggae/dancehall misclassification became obvious.

In [41]:
# --- Historical exploration: official genre + full niche-combination counts ---
#
# genre_combo_counts = (
#     songs
#     .groupby(
#         ["genre", "niche_genres"],
#         dropna=False,
#     )
#     .size()
#     .reset_index(name="count")
#     .sort_values("count", ascending=False)
# )
#
# display(genre_combo_counts.head(50))
#
# genre_combo_counts.to_csv(
#     DATA_PROCESSED / "genre_combo_counts.csv",
#     index=False,
# )

In [42]:
# --- Historical exploration: initial "covered vs uncovered" override audit ---
#
# # Important lesson:
# # "Not explicitly covered by an override" does NOT mean "Other."
# # Unmapped niche tags fall back to the mapped official broad genre.
#
# override_tags = set().union(*GENRE_VOTE_TAGS.values())
#
# covered_tags = all_niche_tags & override_tags
# uncovered_tags = all_niche_tags - override_tags
#
# print("Total niche genres:", len(all_niche_tags))
# print("Covered by explicit votes:", len(covered_tags))
# print("Using fallback/inherit/ignore logic:", len(uncovered_tags))

In [43]:
# --- Historical exploration: dominant official genre by niche tag ---
#
# tag_genre_counts = (
#     songs[["genre_original", "niche_genre_list"]]
#     .explode("niche_genre_list")
#     .dropna(subset=["niche_genre_list"])
#     .assign(
#         niche_genre=lambda x:
#         x["niche_genre_list"]
#         .str.lower()
#         .str.strip()
#     )
#     .groupby(["niche_genre", "genre_original"])
#     .size()
#     .reset_index(name="count")
# )
#
# tag_totals = (
#     tag_genre_counts
#     .groupby("niche_genre")["count"]
#     .sum()
#     .rename("total")
# )
#
# dominant_genres = (
#     tag_genre_counts
#     .sort_values("count", ascending=False)
#     .drop_duplicates("niche_genre")
#     .merge(tag_totals, on="niche_genre")
# )
#
# dominant_genres["dominant_share"] = (
#     dominant_genres["count"]
#     / dominant_genres["total"]
# )
#
# # Practical review screen used during exploration:
# # high-volume tags + dominant share below 80%.
# # Low agreement alone does NOT mean the tag is wrong.

In [44]:
# --- Historical exploration: rebuilding genre_tag_audit.csv ---
#
# exploded_genres = (
#     songs[
#         [
#             "name",
#             "artists",
#             "genre_original",
#             "niche_genre_list",
#         ]
#     ]
#     .explode("niche_genre_list")
#     .dropna(subset=["niche_genre_list"])
#     .rename(columns={"niche_genre_list": "niche_genre"})
# )
#
# exploded_genres["niche_genre"] = (
#     exploded_genres["niche_genre"]
#     .str.lower()
#     .str.strip()
# )
#
# genre_distribution = (
#     exploded_genres
#     .groupby(["niche_genre", "genre_original"])
#     .size()
#     .unstack(fill_value=0)
# )
#
# genre_shares = genre_distribution.div(
#     genre_distribution.sum(axis=1),
#     axis=0,
# )
#
# genre_distribution.columns = [
#     f"{col}_count"
#     for col in genre_distribution.columns
# ]
#
# genre_shares.columns = [
#     f"{col}_share"
#     for col in genre_shares.columns
# ]
#
# genre_audit = pd.concat(
#     [genre_distribution, genre_shares],
#     axis=1,
# )
#
# genre_audit["total_count"] = (
#     genre_distribution.sum(axis=1)
# )
#
# official_counts = (
#     exploded_genres
#     .groupby(["niche_genre", "genre_original"])
#     .size()
#     .reset_index(name="count")
# )
#
# official_counts["rank"] = (
#     official_counts
#     .groupby("niche_genre")["count"]
#     .rank(method="first", ascending=False)
# )
#
# top1 = (
#     official_counts[official_counts["rank"] == 1]
#     .set_index("niche_genre")
# )
#
# top2 = (
#     official_counts[official_counts["rank"] == 2]
#     .set_index("niche_genre")
# )
#
# genre_audit["dominant_genre"] = top1["genre_original"]
# genre_audit["dominant_share"] = (
#     top1["count"] / genre_audit["total_count"]
# )
#
# genre_audit["second_genre"] = top2["genre_original"]
# genre_audit["second_share"] = (
#     top2["count"] / genre_audit["total_count"]
# )
#
#
# def get_examples(group, n=3):
#     examples = (
#         group[["name", "artists"]]
#         .drop_duplicates()
#         .head(n)
#     )
#
#     return " | ".join(
#         examples["name"].astype(str)
#         + " — "
#         + examples["artists"].astype(str)
#     )
#
#
# examples = (
#     exploded_genres
#     .groupby("niche_genre")[["name", "artists"]]
#     .apply(get_examples)
#     .rename("examples")
# )
#
# genre_audit = (
#     genre_audit
#     .join(examples)
#     .reset_index()
#     .sort_values("total_count", ascending=False)
# )
#
# genre_audit.to_csv(
#     DATA_PROCESSED / "genre_tag_audit.csv",
#     index=False,
# )

## Current stopping point

At this stage the notebook has:

- a clean Billboard popularity backbone,
- a memory-conscious 550K metadata load,
- documented rejection of the unreliable Spotify-derived `year` field,
- a reproducible 15-category genre taxonomy,
- a separate `religious_music` feature,
- a canonical Spotify row-selection rule for repeated title + artist entries,
- Pass 1 strict exact title + artist matching,
- Pass 2 exact title + lead artist matching for explicit `Featuring ...` credits,
- and a clean remaining-unmatched diagnostic for deciding the next rule.

**Recommended next action:** run Section 13 from top to bottom and inspect the Pass-2 count plus the remaining-unmatched breakdown. Only then decide whether the next pass should address `With`, artist aliases, title versions, or constrained fuzzy matching.
